In [2]:
import anndata
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import scanpy as sc
import seaborn as sns
import shutil
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture

from helpers import *

import warnings
warnings.filterwarnings("ignore")
sc.settings.verbosity = 0

In [3]:
# (1) Use Isocortex or HPF, (2) Use 25×25 or 50×50 grids
ROI = "Isocortex"
grid_len = "25_by_25"
grid_len_num = int(grid_len.split("_")[0])

# File paths
data_path_wt = f"../data/MERSCOPE_WT_1/processed_data/"
data_path_ad = f"../data/MERSCOPE_AD_1/processed_data/"
comparison_path = f"../output/MERSCOPE_WT_AD_comparison/"
output_path = f"../output/MERSCOPE_WT_AD_comparison/" + f"neuropil_subdomains_{ROI}_{grid_len_num}/"
# shutil.rmtree(output_path, ignore_errors=True)
os.makedirs(output_path, exist_ok=True)

# Plot settings
plot_settings = {"Isocortex": {"25_by_25": {"figure_size": (12.5, 6.5), "spot_size": 25}, "50_by_50": {"figure_size": (12.5, 5.5), "spot_size": 80}},
                 "HPF": {"25_by_25": {"figure_size": (10, 7), "spot_size": 18}, "50_by_50": {"figure_size": (10, 7), "spot_size": 36}}}

# Benchmark clustering settings
k_range = range(2, 10)
random_state = 42
batch_size = 1000
n_seeds = 5
stability_seeds = [0, 42, 123, 456, 789][:n_seeds]

In [4]:
# Recover 50×50 grid annotation
def recover_spots(spots, roi, batch = "MERSCOPE_WT_1"):
    
    # Read reference spots
    spots_ref = sc.read_h5ad(comparison_path + f"neuropil_subdomains_spots_{roi}.h5ad")
    
    # Select spots
    if roi == "Isocortex":
        if batch == "MERSCOPE_WT_1":
            select_mask = spots.obs["region_labels"].isin([1, 3, 15, 12, 13, 23])
        elif batch == "MERSCOPE_AD_1":
            select_mask = spots.obs["region_labels"].isin([2, 5, 11, 17, 26])
        spots_select = spots[select_mask].copy()
    elif roi == "HPF":
        select_mask = spots.obs["brain_area"].isin(["HPF-CA", "HPF-DG", "HPF-SR"])
        spots_select = spots[select_mask].copy()
    
    # Recover annotation
    labels_df = pd.DataFrame({"global_x": spots_ref.obs["global_x"].to_numpy(), "global_y": spots_ref.obs["global_y"].to_numpy(), "layer_labels": spots_ref.obs["layer_labels"].to_numpy()}).reset_index(drop=True)
    ref_xy = labels_df[["global_x", "global_y"]].to_numpy()
    tree = cKDTree(ref_xy)
    if batch == "MERSCOPE_WT_1":
        spots_xy = spots_select.obs[["global_y_new", "global_x_new"]].to_numpy()
    elif batch == "MERSCOPE_AD_1":
        spots_select.obs["global_x_new"] += 12000
        spots_select.obs["global_y_new"] += 7200
        spots_xy = spots_select.obs[["global_x_new", "global_y_new"]].to_numpy()
    _, nn_idx = tree.query(spots_xy, k=1)
    spots_select = spots_select.copy()
    spots_select.obs["layer_labels"] = labels_df.loc[nn_idx, "layer_labels"].to_numpy()
    
    spots.obs["layer_labels"] = None
    spots.obs.loc[select_mask, "layer_labels"] = spots_select.obs["layer_labels"].to_numpy()
    return spots

In [8]:
# ==================== Read data ==================== #

# Spots
if grid_len == "25_by_25":
    
    spots = sc.read_h5ad(comparison_path + f"neuropil_subdomains_spots_{ROI}.h5ad")
    spots = spots[(spots.obs["layer_labels"].notna()) & (spots.obs["layer_labels"] != "Others")].copy()
    
elif grid_len == "50_by_50":
    
    spots_wt = sc.read_h5ad(data_path_wt + "spots.h5ad")
    spots_ad = sc.read_h5ad(data_path_ad + "spots.h5ad")
    
    spots_wt = recover_spots(spots_wt, ROI, batch = "MERSCOPE_WT_1")
    spots_ad = recover_spots(spots_ad, ROI, batch = "MERSCOPE_AD_1")
    
    spots_wt.obs["global_x"] = spots_wt.obs["global_y_new"].copy()
    spots_wt.obs["global_y"] = spots_wt.obs["global_x_new"].copy()

    spots_ad.obs["global_x"] = spots_ad.obs["global_x_new"].copy()
    spots_ad.obs["global_y"] = spots_ad.obs["global_y_new"].copy()
    spots_ad.obs["global_x"] += 12000
    spots_ad.obs["global_y"] += 7200

    spots_dict = {"MERSCOPE_WT_1": spots_wt, "MERSCOPE_AD_1": spots_ad}
    spots = anndata.concat(spots_dict, axis = 0, merge = "same", label = "batch")
    spots = spots[spots.obs["brain_area"] != "Unknown"].copy()
    spots.obs.loc[spots.obs["layer_labels"] == "Others", "layer_labels"] = None
    
    sc.set_figure_params(scanpy = True, figsize = (10, 7))
    ax = sc.pl.scatter(spots, x="global_x", y="global_y", color="layer_labels", size=45, title=" ", show = False)
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title("")
    for spine in ax.spines.values():
        spine.set_visible(False)
    plt.savefig(output_path + f"ROI.jpeg", dpi = 500, bbox_inches = "tight")
    plt.close()
    
    spots = spots[spots.obs["layer_labels"].notna()].copy()

# Cells
adata = sc.read_h5ad(comparison_path + "neuropil_subdomains_adata.h5ad")

# Granules
# granule_adata = sc.read_h5ad(comparison_path + "neuropil_subdomains_granule_adata.h5ad")
granule_adata = sc.read_h5ad(comparison_path + "granule_adata_tsne.h5ad")

granule_adata.obs["global_x"] = granule_adata.obs["global_x_adjusted"].copy()
granule_adata.obs["global_y"] = granule_adata.obs["global_y_adjusted"].copy()
mask = granule_adata.obs["batch"] == "MERSCOPE_WT_1"
granule_adata.obs.loc[mask, "global_x"] = (6250 - granule_adata.obs.loc[mask, "global_x"])

granule_subtype_df = pd.read_parquet(comparison_path + "granule_subtype_labels_granule_adata_tsne.parquet")
cols_keep = ["sample", "granule_id", "granule_subtype_kmeans", "granule_subtype_manual", "granule_subtype_manual_simple"]
granule_subtype_df = granule_subtype_df[cols_keep].drop_duplicates(["sample", "granule_id"])

granule_adata.obs = granule_adata.obs.reset_index(names="obs_name")
granule_adata.obs = granule_adata.obs.merge(granule_subtype_df,
                                            left_on=["batch", "granule_id"],
                                            right_on=["sample", "granule_id"],
                                            how="left",
                                            validate="one_to_one").set_index("obs_name")

if "sample" in granule_adata.obs.columns:
    granule_adata.obs = granule_adata.obs.drop(columns=["sample"])

granule_adata.obs["granule_subtype_kmeans"] = granule_adata.obs["granule_subtype_kmeans"].astype("category")
granule_adata.obs["granule_subtype_manual"] = granule_adata.obs["granule_subtype_manual"].astype("category")
granule_adata.obs["granule_subtype"] = granule_adata.obs["granule_subtype_manual_simple"].astype("category")

# # Plot check
# sc.pl.scatter(spots, x="global_x", y="global_y", color="layer_labels")
# plt.show()

# sc.pl.scatter(adata, x="global_x", y="global_y", color="brain_area")
# plt.show()

# sc.pl.scatter(granule_adata, x="global_x", y="global_y", color="brain_area")
# plt.show()

In [9]:
pd.crosstab(granule_adata.obs["granule_subtype_kmeans"], granule_adata.obs["batch"])

batch,MERSCOPE_WT_1,MERSCOPE_AD_1
granule_subtype_kmeans,,
0,31537,18391
1,167978,82194
2,21971,27118
3,38425,32029
4,58958,36860
5,67022,48415
6,40508,8526
7,15190,10483
8,44595,39532


In [132]:
# Previous neuron-granule counts
adata_neuron = adata[(adata.obs["brain_area"].str.startswith(ROI)) & (adata.obs["cell_type"].isin(["GABAergic", "Glutamatergic"]))].copy()
_, _, granule_counts_neuron = neuron_embedding_spatial_weight(adata_neuron, granule_adata, radius = 10, sigma = 10, loc_key = ["global_x", "global_y"], gnl_subtype_key = "granule_subtype_kmeans", padding_value = "Others")
print(f"Mean neuron-granule counts: {np.nanmean(granule_counts_neuron):.3f}, Median neuron-granule counts: {np.nanmedian(granule_counts_neuron):.3f}")

Mean neuron-granule counts: 13.178, Median neuron-granule counts: 11.000


In [133]:
# ==================== Main operation ==================== #

spots0 = spots.copy()

# -------------------- Embedding -------------------- #
for embedding_method in ["hard", "soft"]:
    
    spots = spots0.copy()
    
    if embedding_method == "hard":
        
        embeddings, embeddings_features, aux_features, spot_granule_expression = spot_embedding(
            spots=spots,
            granule_adata=granule_adata,
            adata=adata,
            spot_width=grid_len_num,
            spot_height=grid_len_num,
            granule_subtype_key="granule_subtype_kmeans",
            subtype_names=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())],
            granule_count_layer="counts",
            include_soma_features=True,
            smoothing=True,
            smoothing_radius=np.sqrt(2) * grid_len_num + 1,
            smoothing_mode="gaussian"
        )
    
    elif embedding_method == "soft":
        
        embeddings, embeddings_features, aux_features, spot_granule_expression = spot_embedding_granule_subtype_kernel_grid(
            spots=spots,
            granule_adata=granule_adata,
            adata=adata,
            spot_width=grid_len_num,
            spot_height=grid_len_num,
            granule_subtype_key="granule_subtype_kmeans",
            subtype_names=[str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())],
            granule_count_layer="counts",
            include_soma_features=True,
            kernel="exponential",
            sigma=grid_len_num / 2,
            support_radius=grid_len_num,
            normalize_subtype_embedding=False,
            normalize_gene_counts=False,
        )
    
    for aux_key, aux_val in aux_features.items():
        spots.obs[aux_key] = aux_val
    
    # Rough comparison with neuron-granule counts
    granule_counts = spots.obs["granule_count"].to_numpy()
    print(f"Mean spot-granule counts: {np.nanmean(granule_counts):.3f}, Median spot-granule counts: {np.nanmedian(granule_counts):.3f}")
    xmax = np.max([np.max(granule_counts), np.max(granule_counts_neuron)])
    
    plt.figure(figsize=(8, 8))
    plt.hist(granule_counts, bins=50, alpha=0.7, edgecolor="black")
    plt.xlim(right=xmax * 1.1)
    plt.grid(False)
    plt.xlabel("Number of Granules")
    plt.ylabel("Count")
    plt.title(" ")
    plt.savefig(output_path + f"{embedding_method}_granule_counts.jpeg", dpi=500, bbox_inches="tight")
    plt.close()
    
    plt.figure(figsize=(8, 8))
    plt.hist(granule_counts_neuron, bins=50, color="#9b59b6", alpha=0.7, edgecolor="black")
    plt.xlim(right=xmax)
    plt.grid(False)
    plt.xlabel("Number of Granules")
    plt.ylabel("Count")
    plt.title(" ")
    plt.savefig(output_path + "granule_counts_per_neuron.jpeg", dpi=500, bbox_inches="tight")
    plt.close()
    
    # Remove spots with no granules
    mask = (spots.obs["granule_count"] > 0)
    spots = spots[mask].copy()
    embeddings = embeddings[mask].copy()
    spot_granule_expression = spot_granule_expression[mask].copy()

    # -------------------- Clustering -------------------- #
    for mode in ["unnormalized", "normalized"]:
        
        if mode == "normalized":
            row_sums = embeddings.sum(axis=1, keepdims=True)
            X = np.divide(embeddings, row_sums, out=np.zeros_like(embeddings, dtype=float), where=row_sums > 0)
        else:
            X = embeddings
        
        # Justify number of clusters using (1) Minibatch K-Means and (2) K = 5
        print(f"Benchmarking number of clusters using {embedding_method} embedding and {mode} mode... ----------")
        
        results = []
        for k in k_range:
            km = MiniBatchKMeans(n_clusters=k, random_state=random_state, batch_size=batch_size)
            labels = km.fit_predict(X)
            inertia = km.inertia_
            label_list = [labels]
            for seed in stability_seeds[1:]:
                km_r = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=batch_size)
                label_list.append(km_r.fit_predict(X))
            ari_values = []
            for i in range(len(label_list)):
                for j in range(i + 1, len(label_list)):
                    ari_values.append(adjusted_rand_score(label_list[i], label_list[j]))
            ari_stability_mean = np.mean(ari_values) if ari_values else np.nan
            results.append({"n_clusters": k, "inertia": inertia, "ari_stability_mean": ari_stability_mean})
        metrics_df = pd.DataFrame(results)
        metrics_df.to_csv(output_path + f"{embedding_method}_{mode}_benchmark_clustering_results.csv", index=False)
        
        plt.figure(figsize=(8, 8))
        plt.plot(metrics_df["n_clusters"], metrics_df["inertia"], marker="o", markersize=6)
        plt.xlabel("Number of Clusters")
        plt.ylabel("Inertia")
        plt.title(" ")
        plt.savefig(output_path + f"{embedding_method}_{mode}_benchmark_clustering_results.jpeg", dpi=500, bbox_inches="tight")
        plt.close()
        
        for n_clusters in [4, 5]:
            
            print(f"---------- Embedding method: {embedding_method}, Mode: {mode}, Number of clusters: {n_clusters} ----------")
            
            # LDA clustering
            lda = LatentDirichletAllocation(n_components = n_clusters, random_state = 42)
            lda_labels = lda.fit_transform(X)
            spots.obs["subdomain_lda"] = [f"Subdomain {l + 1}" for l in np.argmax(lda_labels, axis = 1)]
            
            # GMM clustering
            gmm = GaussianMixture(n_components = n_clusters, random_state = 42)
            gmm_labels = gmm.fit_predict(X)
            spots.obs["subdomain_gmm"] = [f"Subdomain {l + 1}" for l in gmm_labels]
            
            # K-Means clustering
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
            kmeans_labels = kmeans.fit_predict(X)
            spots.obs["subdomain_kmeans"] = [f"Subdomain {l + 1}" for l in kmeans_labels]
            
            # Minibatch K-Means clustering
            kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1000, random_state=42, n_init=20)
            kmeans_labels = kmeans.fit_predict(X)
            spots.obs["subdomain_minibatch"] = [f"Subdomain {l + 1}" for l in kmeans_labels]

            # -------------------- Visualization -------------------- #
            for cluster_col in ["subdomain_lda", "subdomain_gmm", "subdomain_kmeans", "subdomain_minibatch"]:
                
                # Delete previous uns
                spots.uns.pop(f"{cluster_col}_colors", None)
                
                # Scatter plot
                sc.set_figure_params(scanpy = True, figsize = plot_settings[ROI][grid_len]["figure_size"])
                ax = sc.pl.scatter(spots, x="global_x", y="global_y", color=cluster_col, size=plot_settings[ROI][grid_len]["spot_size"], title=" ", show = False)
                ax.grid(False)
                ax.set_xticks([])
                ax.set_yticks([])
                ax.set_xlabel("")
                ax.set_ylabel("")
                ax.set_title("")
                for spine in ax.spines.values():
                    spine.set_visible(False)
                plt.savefig(output_path + f"{n_clusters}_{embedding_method}_{mode}_{cluster_col.split('_')[1]}.jpeg", dpi = 500, bbox_inches = "tight")
                plt.close()
                
                for cluster in spots.obs[cluster_col].unique():
                    sc.set_figure_params(scanpy = True, figsize = plot_settings[ROI][grid_len]["figure_size"])
                    ax = sc.pl.scatter(spots[spots.obs[cluster_col] == cluster], x="global_x", y="global_y", color=cluster_col, size=plot_settings[ROI][grid_len]["spot_size"], title=" ", show = False)
                    ax.grid(False)
                    ax.set_xticks([])
                    ax.set_yticks([])
                    ax.set_xlabel("")
                    ax.set_ylabel("")
                    ax.set_title("")
                    for spine in ax.spines.values():
                        spine.set_visible(False)
                    plt.savefig(output_path + f"{n_clusters}_{embedding_method}_{mode}_{cluster_col.split('_')[1]}_{cluster}.jpeg", dpi = 500, bbox_inches = "tight")
                    plt.close()
                    
                # Embedding heatmap (mean value)
                subtype_labels = [str(i) for i in range(granule_adata.obs["granule_subtype_kmeans"].nunique())]
                desired_order = [f"Subdomain {i + 1}" for i in range(n_clusters)]

                embedding_df = pd.DataFrame(X, columns = subtype_labels)
                embedding_df[cluster_col] = spots.obs[cluster_col].values
                cluster_means = embedding_df.groupby(cluster_col).mean()
                cluster_means = cluster_means.loc[desired_order, subtype_labels]
                
                plt.figure(figsize = (5, 6))
                sns.heatmap(cluster_means.T, cmap = "Reds", annot = False, linewidths = 0.5, linecolor = "lightgray", xticklabels = True, yticklabels = True)
                plt.grid(False)
                plt.title(" ")
                plt.xlabel("Neuropil Subdomain")
                plt.ylabel("Granule Subtype")
                plt.savefig(output_path + f"{n_clusters}_{embedding_method}_{mode}_heatmap_{cluster_col.split('_')[1]}.jpeg", dpi = 500, bbox_inches = "tight")
                plt.close()
                
                # Embedding heatmap (log2FC)
                if mode == "normalized":
                
                    global_means = embedding_df[subtype_labels].mean(axis=0)
                    fold_mat = cluster_means.divide(global_means, axis=1).T
                    eps = 1e-8
                    log2fc_mat = np.log2((cluster_means + eps).divide(global_means + eps, axis=1)).T
                    
                    vmax = np.nanmax(np.abs(log2fc_mat.values))
                    vmin = -vmax
                    plt.figure(figsize=(5, 6))
                    sns.heatmap(log2fc_mat, cmap="bwr", center=0, vmin=vmin, vmax=vmax, linewidths=0.5, linecolor="lightgray")
                    plt.grid(False)
                    plt.title(" ")
                    plt.xlabel("Neuropil Subdomain")
                    plt.ylabel("Granule Subtype")
                    plt.savefig(output_path + f"{n_clusters}_{embedding_method}_{mode}_heatmap_log2fc_{cluster_col.split('_')[1]}.jpeg", dpi=500, bbox_inches="tight")
                    plt.close()

Mean spot-granule counts: 25.204, Median spot-granule counts: 22.187
Benchmarking number of clusters using hard embedding and unnormalized mode... ----------
---------- Embedding method: hard, Mode: unnormalized, Number of clusters: 4 ----------
---------- Embedding method: hard, Mode: unnormalized, Number of clusters: 5 ----------
Benchmarking number of clusters using hard embedding and normalized mode... ----------
---------- Embedding method: hard, Mode: normalized, Number of clusters: 4 ----------
---------- Embedding method: hard, Mode: normalized, Number of clusters: 5 ----------
Mean spot-granule counts: 24.054, Median spot-granule counts: 20.649
Benchmarking number of clusters using soft embedding and unnormalized mode... ----------
---------- Embedding method: soft, Mode: unnormalized, Number of clusters: 4 ----------
---------- Embedding method: soft, Mode: unnormalized, Number of clusters: 5 ----------
Benchmarking number of clusters using soft embedding and normalized mode.

In [ ]:
# Evaluate ARI against K-Means on (1) granule expression and (2) extrasomatic expression

for data, label in zip([spots.X.copy(), spots.layers["extrasomatic_transcripts"].copy(), spot_granule_expression], ["total_expr", "extrasomatic_expr", "granule_expr"]):
    if not isinstance(data, np.ndarray):
        data = data.toarray()
    sums = data.sum(axis=1, keepdims=True)
    sums[sums == 0] = 1
    data = data / sums * 1e4
    data = np.log1p(data)
    kmeans = MiniBatchKMeans(n_clusters=K, batch_size=5000, random_state=0, n_init=20)
    kmeans.fit(data)
    spots.obs[f"{label}_kmeans"] = kmeans.labels_.astype(str)
    print(f"ARI between {label} and subdomain: {adjusted_rand_score(spots.obs[f'{label}_kmeans'], spots.obs['subdomain_gmm_wtref']):.6f}")
    
    ax = sc.pl.scatter(spots, x="global_x", y="global_y", color=f"{label}_kmeans", size=5.5, title=f"{label}_kmeans", show=False)
    plt.savefig(output_path + f"{label}_kmeans.png", dpi = 500, bbox_inches = "tight")
    plt.close()